# Выявление поддельных отзывов на бенгальском языке

**Задача:** сравнить две интерпретируемые модели классификации текстов при сильном дисбалансе классов.

**Стек:** Hugging Face Datasets, scikit-learn, pandas, seaborn.

Разбиение выполняется до обучения TF-IDF и любой балансировки, поэтому информация из теста не попадает в признаки обучающей выборки.


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline

SEED = 42


In [ ]:
dataset = load_dataset("shawon95/Bengali-Fake-Review-Dataset", split="train")
data = dataset.to_pandas()

text_column = "Review" if "Review" in data.columns else data.columns[0]
label_column = "Label" if "Label" in data.columns else data.columns[-1]
data = data[[text_column, label_column]].dropna().drop_duplicates(text_column)
data = data.rename(columns={text_column: "text", label_column: "raw_label"})


def normalize_label(value) -> int:
    if isinstance(value, str):
        normalized = value.strip().lower().replace("_", "-")
        if normalized in {"fake", "1", "true"}:
            return 1
        if normalized in {"non-fake", "nonfake", "real", "0", "false"}:
            return 0
        raise ValueError(f"Неизвестная метка: {value!r}")
    return int(value)


def clean_bengali_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^\u0980-\u09ff\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


data["label"] = data["raw_label"].map(normalize_label)
data["text"] = data["text"].map(clean_bengali_text)
data = data[data["text"].str.len() > 0].reset_index(drop=True)

print(data["label"].value_counts().rename(index={0: "real", 1: "fake"}))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"],
)

models = {
    "Complement Naive Bayes": Pipeline(
        [
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20_000)),
            ("classifier", ComplementNB(alpha=0.5)),
        ]
    ),
    "Logistic Regression": Pipeline(
        [
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20_000)),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2_000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    ),
}


In [ ]:
rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    predictions[name] = y_pred
    rows.append(
        {
            "model": name,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_score),
            "pr_auc": average_precision_score(y_test, y_score),
        }
    )

metrics = pd.DataFrame(rows).set_index("model").sort_values("f1", ascending=False)
metrics.round(4)


In [ ]:
best_model_name = metrics.index[0]
best_predictions = predictions[best_model_name]
print(best_model_name)
print(classification_report(y_test, best_predictions, target_names=["real", "fake"]))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    display_labels=["real", "fake"],
    cmap="Blues",
)
plt.title(f"Матрица ошибок: {best_model_name}")
plt.tight_layout()


In [ ]:
best_model = models[best_model_name]
sample = X_test.sample(n=min(10, len(X_test)), random_state=SEED)
sample_predictions = best_model.predict(sample)
pd.DataFrame(
    {
        "text": sample.values,
        "true_label": y_test.loc[sample.index].values,
        "predicted_label": sample_predictions,
    }
)


## Интерпретация

Из-за дисбаланса классов основными метриками являются F1 и PR-AUC для класса `fake`. Простая accuracy может выглядеть высокой даже у модели, которая редко находит поддельные отзывы.
